# 02 - Pretraitement, baseline, Transformer et evaluation (ISLR)

Activer le GPU (Settings > Accelerator). Toutes les seeds sont fixees ; la configuration
de chaque run est ecrite dans `runs/*/config.json`.

Separation PAR SIGNEUR : aucun signeur commun entre train, val et test.

In [ ]:
# Acces au code du projet : soit un dataset Kaggle "lsf-code" (dossier src/ du depot),
# soit un clone GitHub (activer Internet dans les reglages du notebook).
import glob, os, subprocess, sys
CODE_DATASET = "/kaggle/input/lsf-code/src"
REPO = "https://github.com/M-Jel/LSF_project.git"  # A CONFIRMER : nom du depot GitHub
if os.path.exists(CODE_DATASET):
    sys.path.insert(0, CODE_DATASET)
else:
    if not os.path.exists("/kaggle/working/LSF_project"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/LSF_project"], check=True)
    sys.path.insert(0, "/kaggle/working/LSF_project/src")

# Localisation automatique des donnees de la competition asl-signs
ROOT = os.path.dirname(sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))[0])
print("donnees :", ROOT)

In [ ]:
# 1) Cache pretraite (environ 3,2 Go en float16)
CACHE = "/kaggle/working/islr_cache.npz"
if not os.path.exists(CACHE):
    !cd /kaggle/working && PYTHONPATH={sys.path[0]} python -m lsf.data.islr --root {ROOT} --out {CACHE} --workers 4

In [ ]:
# 2) Separation par signeur (a recuperer et versionner : splits/islr_signer_split.json)
import numpy as np
from lsf.data.split import make_signer_split, save_split, describe
z = np.load(CACHE)
split = make_signer_split(z["participant"], z["labels"], val_frac=0.15, test_frac=0.15, seed=0)
os.makedirs("/kaggle/working/splits", exist_ok=True)
SPLIT = "/kaggle/working/splits/islr_signer_split.json"
save_split(split, SPLIT, seed=0, val_frac=0.15, test_frac=0.15)
print(describe(z["participant"], split))

In [ ]:
# 3) Baseline
!cd /kaggle/working && PYTHONPATH={sys.path[0]} python -m lsf.train --cache {CACHE} --split {SPLIT} --model baseline --epochs 30 --seed 0 --out runs/baseline_s0

In [ ]:
# 4) Transformer
!cd /kaggle/working && PYTHONPATH={sys.path[0]} python -m lsf.train --cache {CACHE} --split {SPLIT} --model transformer --epochs 60 --seed 0 --out runs/transformer_s0

In [ ]:
# 5) Evaluation sur les signeurs de test
for run in ["baseline_s0", "transformer_s0"]:
    !cd /kaggle/working && PYTHONPATH={sys.path[0]} python -m lsf.evaluate --cache {CACHE} --ckpt runs/{run}/best.pt --out reports/{run}

In [ ]:
# 6) Courbes d'apprentissage et paires confondues
import json, pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for run in ["baseline_s0", "transformer_s0"]:
    h = pd.DataFrame(json.load(open(f"/kaggle/working/runs/{run}/history.json")))
    h.plot(x="epoch", y="val_top1", ax=ax[0], label=run)
    h.plot(x="epoch", y="train_loss", ax=ax[1], label=run)
ax[0].set_title("top-1 validation"); ax[1].set_title("perte d'entrainement")
plt.savefig("/kaggle/working/reports/curves.png", dpi=120)
pd.read_csv("/kaggle/working/reports/transformer_s0/confused_pairs.csv").head(15)

In [ ]:
# 7) Archive a telecharger (rapports + split + poids). Verifier les regles de la
# competition avant de publier les poids.
!cd /kaggle/working && zip -qr results.zip reports splits runs/*/best.pt runs/*/*.json && ls -lh results.zip